In [1]:
# ============================================================
# IMPORTAÇÕES
# ============================================================

import io
import re
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import confusion_matrix


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def sanitizar_nome(nome):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(nome))


def fig_to_base64(fig):
    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)
    buffer.seek(0)

    imagem_base64 = base64.b64encode(
        buffer.read()
    ).decode("utf-8")

    return imagem_base64


def formatar_count(valor):
    return f"{int(valor):,}".replace(",", ".")


# ============================================================
# FUNÇÃO PRINCIPAL 1x1
# ============================================================

def gerar_relatorio_1x1(
    rank,
    arquivo_dados="creditcard.csv",
    arquivo_scores="1x1_scores.csv",
    pasta_saida=".",
    nome_base_saida="1x1_relat"
):
    """
    Gera relatório HTML completo para a feature 1x1 escolhida pelo rank.

    Entrada:
        rank: posição do ranking em 1x1_scores.csv

    Saída:
        HTML com:
        - métricas principais
        - matriz de confusão no melhor ponto de corte
        - matriz de confusão no ponto médio 0.5
        - matriz ideal
        - boxplot da feature por classe
        - representação 1D por classe real
        - correlação de Spearman entre feature e target
        - responsabilidades GMM no melhor ponto de corte
        - responsabilidades GMM no ponto de corte médio
    """

    # ========================================================
    # LEITURA DOS ARQUIVOS
    # ========================================================

    df = pd.read_csv(arquivo_dados)
    scores_1x1 = pd.read_csv(arquivo_scores)

    # ========================================================
    # DEFINIÇÃO DO TARGET
    # ========================================================

    if "status_fraude" in df.columns:
        target_name = "status_fraude"

    elif "Class" in df.columns:
        df = df.rename(columns={"Class": "status_fraude"})
        target_name = "status_fraude"

    else:
        raise ValueError("Não encontrei a coluna target: 'status_fraude' ou 'Class'.")

    # ========================================================
    # VALIDAÇÕES DO CSV DE SCORES
    # ========================================================

    colunas_necessarias = [
        "Feature",
        "Posicao_Rank",
        "AUC_PR",
        "MCC",
        "Score_Final",
        "Melhor_Ponto_Corte"
    ]

    for coluna in colunas_necessarias:
        if coluna not in scores_1x1.columns:
            raise ValueError(
                f"O arquivo {arquivo_scores} precisa ter a coluna '{coluna}'."
            )

    if "Ponto_Corte_Medio" not in scores_1x1.columns:
        scores_1x1["Ponto_Corte_Medio"] = 0.5

    if rank not in scores_1x1["Posicao_Rank"].values:
        raise ValueError(
            f"Rank {rank} não encontrado. "
            f"Ranks disponíveis: 1 até {scores_1x1['Posicao_Rank'].max()}."
        )

    # ========================================================
    # PEGAR FEATURE PELO RANK
    # ========================================================

    linha_rank = scores_1x1.loc[
        scores_1x1["Posicao_Rank"] == rank
    ].iloc[0]

    feature = linha_rank["Feature"]

    melhor_ponto_corte = float(linha_rank["Melhor_Ponto_Corte"])
    ponto_corte_medio = float(linha_rank["Ponto_Corte_Medio"])

    auc_pr = float(linha_rank["AUC_PR"])
    mcc = float(linha_rank["MCC"])

    if "Log_Loss_Norm" in linha_rank.index:
        log_loss_norm = float(linha_rank["Log_Loss_Norm"])
    else:
        log_loss_norm = 1 / (1 + float(linha_rank["Log_Loss"]))

    score_final = float(linha_rank["Score_Final"])

    if "Diferenca_Neg_Log_Veross" in linha_rank.index:
        diferenca_neg_log_veross = float(linha_rank["Diferenca_Neg_Log_Veross"])

    elif (
        "Neg_Log_Veross_Com_Rotulo" in linha_rank.index
        and "Neg_Log_Veross_GMM" in linha_rank.index
    ):
        diferenca_neg_log_veross = (
            float(linha_rank["Neg_Log_Veross_Com_Rotulo"])
            - float(linha_rank["Neg_Log_Veross_GMM"])
        )

    else:
        diferenca_neg_log_veross = np.nan

    # ========================================================
    # PREPARAÇÃO DOS DADOS
    # ========================================================

    temp = df[[feature, target_name]].dropna().copy()

    X = temp[[feature]]
    y_real = temp[target_name].astype(int)

    # ========================================================
    # ESCALONAMENTO
    # ========================================================

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # ========================================================
    # TREINAMENTO DA GMM 1D
    # ========================================================

    gmm = GaussianMixture(
        n_components=2,
        covariance_type="full",
        random_state=42,
        n_init=3,
        reg_covar=1e-6
    )

    gmm.fit(X_scaled)

    # ========================================================
    # IDENTIFICAR CLUSTER MAIS ASSOCIADO À FRAUDE
    # ========================================================

    clusters = gmm.predict(X_scaled)

    ct = pd.crosstab(
        clusters,
        y_real
    )

    if 1 not in ct.columns:
        raise ValueError("A classe fraude, valor 1, não foi encontrada no target.")

    cluster_fraude = ct[1].idxmax()

    # ========================================================
    # RESPONSABILIDADES / PROBABILIDADES
    # ========================================================

    responsabilidades = gmm.predict_proba(X_scaled)

    probabilidades = responsabilidades[:, cluster_fraude]

    probabilidades = np.clip(
        probabilidades,
        1e-15,
        1 - 1e-15
    )

    # ========================================================
    # FUNÇÕES INTERNAS: MATRIZ DE CONFUSÃO
    # ========================================================

    def gerar_matriz_confusao(y_real, probabilidades, threshold):
        y_pred = (probabilidades >= threshold).astype(int)

        cm = confusion_matrix(
            y_real,
            y_pred,
            labels=[0, 1]
        )

        return cm

    def preparar_valores_matriz(cm):
        """
        Entrada sklearn:
            [[TN, FP],
             [FN, TP]]

        Saída visual:
            Linha 1: Real Fraude      -> FN, TP
            Linha 2: Real Não Fraude  -> TN, FP
        """

        tn, fp, fn, tp = cm.ravel()

        total_fraudes = fn + tp
        total_nao_fraudes = tn + fp

        fn_pct = fn / total_fraudes * 100 if total_fraudes != 0 else 0
        tp_pct = tp / total_fraudes * 100 if total_fraudes != 0 else 0

        tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0
        fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes != 0 else 0

        valores = {
            "fn": {
                "pct": fn_pct,
                "count": int(fn),
                "qualidade": 100 - fn_pct
            },
            "tp": {
                "pct": tp_pct,
                "count": int(tp),
                "qualidade": tp_pct
            },
            "tn": {
                "pct": tn_pct,
                "count": int(tn),
                "qualidade": tn_pct
            },
            "fp": {
                "pct": fp_pct,
                "count": int(fp),
                "qualidade": 100 - fp_pct
            }
        }

        return valores

    def preparar_valores_matriz_ideal(y_real):
        y_real_array = np.asarray(y_real).astype(int)

        total_fraudes = int(np.sum(y_real_array == 1))
        total_nao_fraudes = int(np.sum(y_real_array == 0))

        valores = {
            "fn": {
                "pct": 0.0,
                "count": 0,
                "qualidade": 100.0
            },
            "tp": {
                "pct": 100.0,
                "count": total_fraudes,
                "qualidade": 100.0
            },
            "tn": {
                "pct": 100.0,
                "count": total_nao_fraudes,
                "qualidade": 100.0
            },
            "fp": {
                "pct": 0.0,
                "count": 0,
                "qualidade": 100.0
            }
        }

        return valores

    def cor_por_qualidade(q):
        if q >= 95:
            return "cell q95"
        elif q >= 85:
            return "cell q85"
        elif q >= 70:
            return "cell q70"
        elif q >= 50:
            return "cell q50"
        elif q >= 30:
            return "cell q30"
        else:
            return "cell q10"

    def gerar_html_matriz(titulo, valores, matriz_ideal=False):

        if matriz_ideal:
            desc_fn = "Erro ideal: nenhuma fraude perdida"
            desc_tp = "Acerto ideal: fraudes detectadas"
            desc_tn = "Acerto ideal: não fraudes corretas"
            desc_fp = "Erro ideal: nenhum falso alerta"
        else:
            desc_fn = "Erro: fraude perdida"
            desc_tp = "Acerto: fraude detectada"
            desc_tn = "Acerto: não fraude"
            desc_fp = "Erro: falso alerta"

        html = f"""
        <section class="matrix-card">
            <h2>{titulo}</h2>

            <div class="matrix-area">

                <div class="matrix-wrapper">

                    <div class="corner"></div>
                    <div class="x-label">Pred Não Fraude</div>
                    <div class="x-label">Pred Fraude</div>

                    <div class="y-label">Real Fraude</div>

                    <div class="{cor_por_qualidade(valores['fn']['qualidade'])}">
                        <div class="pct">{valores['fn']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['fn']['count'])})</div>
                        <div class="cell-desc">{desc_fn}</div>
                    </div>

                    <div class="{cor_por_qualidade(valores['tp']['qualidade'])}">
                        <div class="pct">{valores['tp']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['tp']['count'])})</div>
                        <div class="cell-desc">{desc_tp}</div>
                    </div>

                    <div class="y-label">Real Não Fraude</div>

                    <div class="{cor_por_qualidade(valores['tn']['qualidade'])}">
                        <div class="pct">{valores['tn']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['tn']['count'])})</div>
                        <div class="cell-desc">{desc_tn}</div>
                    </div>

                    <div class="{cor_por_qualidade(valores['fp']['qualidade'])}">
                        <div class="pct">{valores['fp']['pct']:.2f}%</div>
                        <div class="count">({formatar_count(valores['fp']['count'])})</div>
                        <div class="cell-desc">{desc_fp}</div>
                    </div>

                </div>

                <div class="legend">
                    <div class="legend-title">Qualidade</div>
                    <div class="colorbar"></div>
                    <div class="legend-label-top">Melhor</div>
                    <div class="legend-label-bottom">Pior</div>
                </div>

            </div>
        </section>
        """

        return html

    # ========================================================
    # BOXPLOT 1D POR CLASSE
    # ========================================================

    def gerar_boxplot_1d_base64(
        df_plot,
        feature,
        target_name
    ):
        dados_nao_fraude = df_plot.loc[
            df_plot[target_name] == 0,
            feature
        ].dropna()

        dados_fraude = df_plot.loc[
            df_plot[target_name] == 1,
            feature
        ].dropna()

        fig, ax = plt.subplots(figsize=(10, 6))

        box = ax.boxplot(
            [
                dados_nao_fraude,
                dados_fraude
            ],
            labels=[
                f"Não Fraude ({len(dados_nao_fraude):,})".replace(",", "."),
                f"Fraude ({len(dados_fraude):,})".replace(",", ".")
            ],
            patch_artist=True,
            showfliers=True
        )

        cores = ["#2ecc71", "#facc15"]

        for patch, cor in zip(box["boxes"], cores):
            patch.set_facecolor(cor)
            patch.set_alpha(0.72)
            patch.set_linewidth(2)

        for median in box["medians"]:
            median.set_color("#111827")
            median.set_linewidth(2.3)

        for whisker in box["whiskers"]:
            whisker.set_color("#334155")
            whisker.set_linewidth(1.6)

        for cap in box["caps"]:
            cap.set_color("#334155")
            cap.set_linewidth(1.6)

        for flier in box["fliers"]:
            flier.set_marker("o")
            flier.set_markerfacecolor("#64748b")
            flier.set_markeredgecolor("#64748b")
            flier.set_alpha(0.20)
            flier.set_markersize(2.5)

        ax.set_ylabel(
            feature,
            fontsize=14,
            fontweight="bold"
        )

        ax.grid(
            axis="y",
            alpha=0.25
        )

        plt.tight_layout()

        return fig_to_base64(fig)

    # ========================================================
    # REPRESENTAÇÃO 1D COM 100% DOS DADOS
    # ========================================================

    def gerar_representacao_1d_base64(
        df_plot,
        feature,
        target_name
    ):
        dados_nao_fraude = df_plot.loc[
            df_plot[target_name] == 0,
            feature
        ].dropna()

        dados_fraude = df_plot.loc[
            df_plot[target_name] == 1,
            feature
        ].dropna()

        rng = np.random.default_rng(42)

        y_nao_fraude = rng.normal(
            loc=0.0,
            scale=0.025,
            size=len(dados_nao_fraude)
        )

        y_fraude = rng.normal(
            loc=1.0,
            scale=0.035,
            size=len(dados_fraude)
        )

        fig, ax = plt.subplots(figsize=(11, 4.8))

        ax.scatter(
            dados_nao_fraude,
            y_nao_fraude,
            s=6,
            alpha=0.06,
            color="#2ecc71",
            rasterized=True
        )

        ax.scatter(
            dados_fraude,
            y_fraude,
            s=32,
            alpha=0.88,
            color="#facc15",
            edgecolors="#111827",
            linewidths=0.25,
            rasterized=True
        )

        ax.set_yticks([0, 1])
        ax.set_yticklabels(
            ["Não Fraude", "Fraude"],
            fontsize=12,
            fontweight="bold"
        )

        ax.set_xlabel(
            feature,
            fontsize=14,
            fontweight="bold"
        )

        ax.set_ylabel(
            "Classe Real",
            fontsize=13,
            fontweight="bold"
        )

        ax.grid(alpha=0.22)

        legenda_nao_fraude = plt.Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markersize=8,
            markerfacecolor="#2ecc71",
            markeredgecolor="#15803d",
            label=f"Não Fraude ({len(dados_nao_fraude):,})".replace(",", ".")
        )

        legenda_fraude = plt.Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markersize=8,
            markerfacecolor="#facc15",
            markeredgecolor="#111827",
            label=f"Fraude ({len(dados_fraude):,})".replace(",", ".")
        )

        ax.legend(
            handles=[
                legenda_nao_fraude,
                legenda_fraude
            ],
            title="Classe Real",
            loc="best",
            frameon=True
        )

        plt.tight_layout()

        return fig_to_base64(fig)

    # ========================================================
    # CORRELAÇÃO DE SPEARMAN ENTRE FEATURE E TARGET
    # ========================================================

    def gerar_matriz_spearman_1d_base64(
        df_plot,
        feature,
        target_name
    ):
        dados_corr = df_plot[
            [
                feature,
                target_name
            ]
        ].copy()

        dados_corr = dados_corr.rename(
            columns={
                target_name: "Fraude"
            }
        )

        corr = dados_corr.corr(
            method="spearman"
        )

        fig, ax = plt.subplots(figsize=(6.5, 5.5))

        im = ax.imshow(
            corr.values,
            cmap="coolwarm",
            vmin=-1,
            vmax=1
        )

        cbar = plt.colorbar(
            im,
            ax=ax
        )

        cbar.set_label(
            "Correlação de Spearman",
            fontsize=11,
            fontweight="bold"
        )

        labels = corr.columns.tolist()

        ax.set_xticks(np.arange(len(labels)))
        ax.set_yticks(np.arange(len(labels)))

        ax.set_xticklabels(
            labels,
            fontsize=12,
            fontweight="bold",
            rotation=35,
            ha="right"
        )

        ax.set_yticklabels(
            labels,
            fontsize=12,
            fontweight="bold"
        )

        for i in range(len(labels)):
            for j in range(len(labels)):
                valor = corr.values[i, j]

                ax.text(
                    j,
                    i,
                    f"{valor:.3f}",
                    ha="center",
                    va="center",
                    color="black",
                    fontsize=13,
                    fontweight="bold"
                )

        plt.tight_layout()

        return fig_to_base64(fig)

    # ========================================================
    # RESPONSABILIDADE GMM EM 1D COM CURVA DE NÍVEL DO CORTE
    # ========================================================

    def calcular_pontos_corte_1d(
        x_grid,
        prob_grid,
        threshold
    ):
        """
        Calcula os pontos aproximados onde:
            probabilidade_GMM = threshold

        Retorna valores na escala original da feature.
        """

        diff = prob_grid - threshold

        idx = np.where(
            np.sign(diff[:-1]) != np.sign(diff[1:])
        )[0]

        pontos = []

        for i in idx:
            x1 = x_grid[i]
            x2 = x_grid[i + 1]

            y1 = diff[i]
            y2 = diff[i + 1]

            if y2 == y1:
                pontos.append(x1)
            else:
                x_cross = x1 - y1 * (x2 - x1) / (y2 - y1)
                pontos.append(x_cross)

        return pontos

    def gerar_responsabilidades_1d_base64(
        df_plot,
        feature,
        target_name,
        scaler,
        gmm,
        cluster_fraude,
        probabilidades,
        threshold
    ):
        """
        Gráfico 1D das responsabilidades estimadas pela GMM.

        - eixo X = feature original
        - eixo Y = responsabilidade GMM para fraude
        - linha horizontal = ponto de corte
        - linhas verticais = pontos onde responsabilidade = corte
        """

        y_real_array = df_plot[target_name].astype(int).to_numpy()

        mask_nao_fraude = y_real_array == 0
        mask_fraude = y_real_array == 1

        x_original = df_plot[feature].to_numpy()

        x_min = np.min(x_original)
        x_max = np.max(x_original)

        margem = 0.05 * (x_max - x_min)

        x_grid = np.linspace(
            x_min - margem,
            x_max + margem,
            1200
        )

        x_grid_df = pd.DataFrame(
            {
                feature: x_grid
            }
        )

        x_grid_scaled = scaler.transform(x_grid_df)

        prob_grid = gmm.predict_proba(
            x_grid_scaled
        )[:, cluster_fraude]

        pontos_corte = calcular_pontos_corte_1d(
            x_grid=x_grid,
            prob_grid=prob_grid,
            threshold=threshold
        )

        fig, ax = plt.subplots(figsize=(11, 6.2))

        ax.scatter(
            x_original[mask_nao_fraude],
            probabilidades[mask_nao_fraude],
            s=7,
            alpha=0.15,
            color="#60a5fa",
            rasterized=True
        )

        ax.scatter(
            x_original[mask_fraude],
            probabilidades[mask_fraude],
            s=34,
            alpha=0.90,
            color="#facc15",
            edgecolors="#111827",
            linewidths=0.25,
            rasterized=True
        )

        ax.plot(
            x_grid,
            prob_grid,
            color="#dc2626",
            linewidth=2.3,
            label="Responsabilidade estimada pela GMM"
        )

        ax.axhline(
            y=threshold,
            color="#111827",
            linestyle="--",
            linewidth=2.0,
            label="Ponto de corte"
        )

        for ponto in pontos_corte:
            ax.axvline(
                x=ponto,
                color="#111827",
                linestyle=":",
                linewidth=1.8,
                alpha=0.95
            )

        ax.set_title(
            f"Corte = {threshold:.6f}",
            fontsize=13,
            fontweight="bold",
            pad=12
        )

        ax.set_xlabel(
            feature,
            fontsize=14,
            fontweight="bold"
        )

        ax.set_ylabel(
            "Responsabilidade GMM para fraude",
            fontsize=13,
            fontweight="bold"
        )

        ax.set_ylim(-0.03, 1.03)

        ax.grid(alpha=0.22)

        legenda_nao_fraude = plt.Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markersize=8,
            markerfacecolor="#60a5fa",
            markeredgecolor="#1e3a8a",
            label=f"Não Fraude real ({mask_nao_fraude.sum():,})".replace(",", ".")
        )

        legenda_fraude = plt.Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            markersize=8,
            markerfacecolor="#facc15",
            markeredgecolor="#111827",
            label=f"Fraude real ({mask_fraude.sum():,})".replace(",", ".")
        )

        legenda_corte_vertical = plt.Line2D(
            [0],
            [0],
            linestyle=":",
            linewidth=1.8,
            color="#111827",
            label="Ponto(s) onde responsabilidade = corte"
        )

        handles, labels = ax.get_legend_handles_labels()

        ax.legend(
            handles=[
                legenda_nao_fraude,
                legenda_fraude,
                handles[0],
                handles[1],
                legenda_corte_vertical
            ],
            title="Legenda",
            loc="best",
            frameon=True
        )

        plt.tight_layout()

        return fig_to_base64(fig)

    # ========================================================
    # GERAR MATRIZES DE CONFUSÃO
    # ========================================================

    cm_melhor = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=probabilidades,
        threshold=melhor_ponto_corte
    )

    cm_medio = gerar_matriz_confusao(
        y_real=y_real,
        probabilidades=probabilidades,
        threshold=ponto_corte_medio
    )

    valores_melhor = preparar_valores_matriz(cm_melhor)
    valores_medio = preparar_valores_matriz(cm_medio)
    valores_ideal = preparar_valores_matriz_ideal(y_real)

    # ========================================================
    # GERAR GRÁFICOS
    # ========================================================

    boxplot_1d_base64 = gerar_boxplot_1d_base64(
        df_plot=temp,
        feature=feature,
        target_name=target_name
    )

    representacao_1d_base64 = gerar_representacao_1d_base64(
        df_plot=temp,
        feature=feature,
        target_name=target_name
    )

    matriz_spearman_base64 = gerar_matriz_spearman_1d_base64(
        df_plot=temp,
        feature=feature,
        target_name=target_name
    )

    grafico_responsabilidades_melhor_base64 = gerar_responsabilidades_1d_base64(
        df_plot=temp,
        feature=feature,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        probabilidades=probabilidades,
        threshold=melhor_ponto_corte
    )

    grafico_responsabilidades_medio_base64 = gerar_responsabilidades_1d_base64(
        df_plot=temp,
        feature=feature,
        target_name=target_name,
        scaler=scaler,
        gmm=gmm,
        cluster_fraude=cluster_fraude,
        probabilidades=probabilidades,
        threshold=ponto_corte_medio
    )

    # ========================================================
    # HTML DAS MATRIZES
    # ========================================================

    html_melhor = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {feature} - Melhor Ponto de Corte",
        valores=valores_melhor
    )

    html_medio = gerar_html_matriz(
        titulo=f"Matriz de Confusão (%) - {feature} - Ponto de Corte 0.5",
        valores=valores_medio
    )

    html_ideal = gerar_html_matriz(
        titulo="Matriz de Confusão Ideal (%)",
        valores=valores_ideal,
        matriz_ideal=True
    )

    # ========================================================
    # HTML FINAL
    # ========================================================

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">
        <title>Relatório 1x1 - Rank {rank} - {feature}</title>

        <style>
            body {{
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
                margin: 0;
                padding: 32px;
            }}

            .container {{
                max-width: 1250px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                margin-bottom: 28px;
                color: #020617;
            }}

            .info-box {{
                background: #ffffff;
                border-radius: 16px;
                padding: 20px 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .info-grid {{
                display: grid;
                grid-template-columns: repeat(3, 1fr);
                gap: 14px;
                margin-top: 14px;
            }}

            .info-item {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                padding: 12px 14px;
            }}

            .info-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 6px;
            }}

            .info-value {{
                font-size: 18px;
                font-weight: 800;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
            }}

            .matrix-card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .matrix-card h2,
            .plot-card h2 {{
                text-align: center;
                margin-top: 0;
                margin-bottom: 24px;
                color: #020617;
                font-size: 22px;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{
                background: #08306b;
            }}

            .q85 {{
                background: #08519c;
            }}

            .q70 {{
                background: #2171b5;
            }}

            .q50 {{
                background: #6baed6;
            }}

            .q30 {{
                background: #c6dbef;
            }}

            .q10 {{
                background: #eff6ff;
            }}

            .legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .legend-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .colorbar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(
                    to bottom,
                    #08306b 0%,
                    #08519c 18%,
                    #2171b5 36%,
                    #6baed6 58%,
                    #c6dbef 78%,
                    #eff6ff 100%
                );
                border: 1px solid #cbd5e1;
            }}

            .legend-label-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .legend-label-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .plot-card {{
                background: white;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .plot-img {{
                display: block;
                max-width: 100%;
                margin: 0 auto;
                border-radius: 12px;
                border: 1px solid #e2e8f0;
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Relatório 1x1 - Rank {rank} - Feature {feature}</h1>

            <div class="info-box">
                <div class="info-grid">

                    <div class="info-item">
                        <div class="info-label">Rank</div>
                        <div class="info-value">{rank}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Feature</div>
                        <div class="info-value">{feature}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Melhor Ponto de Corte</div>
                        <div class="info-value">{melhor_ponto_corte:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Ponto de Corte Médio</div>
                        <div class="info-value">{ponto_corte_medio:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">AUC-PR</div>
                        <div class="info-value">{auc_pr:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">MCC</div>
                        <div class="info-value">{mcc:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Log Loss Norm</div>
                        <div class="info-value">{log_loss_norm:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Score Final</div>
                        <div class="info-value">{score_final:.6f}</div>
                    </div>

                    <div class="info-item">
                        <div class="info-label">Diferença Neg. Log-Veross.</div>
                        <div class="info-value">{diferenca_neg_log_veross:.6f}</div>
                    </div>

                </div>
            </div>

            {html_melhor}

            {html_medio}

            {html_ideal}

            <section class="plot-card">
                <h2>Boxplot por Classe - {feature}</h2>
                <img
                    class="plot-img"
                    src="data:image/png;base64,{boxplot_1d_base64}"
                    alt="Boxplot por Classe - {feature}"
                >
            </section>

            <section class="plot-card">
                <h2>Representação 1D com 100% dos Dados - {feature}</h2>
                <img
                    class="plot-img"
                    src="data:image/png;base64,{representacao_1d_base64}"
                    alt="Representação 1D com 100% dos Dados - {feature}"
                >
            </section>

            <section class="plot-card">
                <h2>Correlação de Spearman entre Feature e Target - {feature} e Fraude</h2>
                <img
                    class="plot-img"
                    src="data:image/png;base64,{matriz_spearman_base64}"
                    alt="Correlação de Spearman entre Feature e Target - {feature} e Fraude"
                >
            </section>

            <section class="plot-card">
                <h2>Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - {feature}</h2>
                <img
                    class="plot-img"
                    src="data:image/png;base64,{grafico_responsabilidades_melhor_base64}"
                    alt="Responsabilidades Estimadas pela GMM - Melhor Ponto de Corte - {feature}"
                >
            </section>

            <section class="plot-card">
                <h2>Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - {feature}</h2>
                <img
                    class="plot-img"
                    src="data:image/png;base64,{grafico_responsabilidades_medio_base64}"
                    alt="Responsabilidades Estimadas pela GMM - Ponto de Corte Médio - {feature}"
                >
            </section>

        </div>
    </body>
    </html>
    """

    # ========================================================
    # SALVAR HTML
    # ========================================================

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    nome_arquivo = (
        f"{nome_base_saida}_rank_{rank}_"
        f"{sanitizar_nome(feature)}.html"
    )

    caminho_html = pasta_saida / nome_arquivo

    caminho_html.write_text(
        html_final,
        encoding="utf-8"
    )

    print(f"HTML gerado com sucesso: {caminho_html.resolve()}")

    return caminho_html

In [2]:
gerar_relatorio_1x1(rank=1)
gerar_relatorio_1x1(rank=2)
gerar_relatorio_1x1(rank=3)




C:\Users\Vitor Craveiro\AppData\Local\Temp\ipykernel_10684\375127209.py:424: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box = ax.boxplot(


HTML gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1x1_relat_rank_1_V17.html


C:\Users\Vitor Craveiro\AppData\Local\Temp\ipykernel_10684\375127209.py:424: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box = ax.boxplot(


HTML gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1x1_relat_rank_2_V12.html


C:\Users\Vitor Craveiro\AppData\Local\Temp\ipykernel_10684\375127209.py:424: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  box = ax.boxplot(


HTML gerado com sucesso: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\1x1_relat_rank_3_V14.html


WindowsPath('1x1_relat_rank_3_V14.html')